In [0]:
# STEP 1 — Read dataset

customer_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/my_files/customers-1000.csv")



In [0]:
customer_df = customer_df.toDF(*[c.replace(" ", "_").lower() for c in customer_df.columns])

display(customer_df)

In [0]:
# STEP 2 — Bronze

customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/my_files/bronze")

bronze_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_files/bronze")



In [0]:
# STEP 3 — Silver

from pyspark.sql.functions import col

silver_df = bronze_df.dropDuplicates(["customer_id"]) \
                     .filter(col("email").isNotNull())

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/my_files/silver")

silver_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_files/silver")


In [0]:

# STEP 4 — Gold

from pyspark.sql.functions import count

gold_df = silver_df.groupBy("country") \
                   .agg(count("customer_id").alias("total_customers"))

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/my_files/gold")